In [2]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="someonecantcode-",
    # Set the wandb project where this run will be logged.
    project="my-awesome-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

acc,▁▆▇▇███▇
loss,█▄▂▁▁▂▁▁
acc,0.82024
loss,0.10501


VAE plan,

Use convolutional resnets to down sample and reconstruct. Bottle neck will use attention and some more convolutional resnets to return our mu and log var.

For loss, simply use L1 MAE reconstruction loss, simple DKL loss with latent to guassian distribution, and LPIPS loss.

Better models, use GANs?

Interesting: <https://arxiv.org/pdf/2605.13565>

In [117]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import lpips

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

lpips_loss_function = lpips.LPIPS(net='vgg').to(device)
lpips_loss_function.requires_grad_(False)

cuda
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\T\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\T\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\T\AppData\Local\Programs\Python\Python312\Lib\site-packages\lpips\weights\v0.1\vgg.pth


LPIPS(
  (scaling_layer): ScalingLayer()
  (net): vgg16(
    (slice1): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
    (slice2): Sequential(
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
    )
    (slice3): Sequential(
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, 

In [154]:
class ResNet(nn.Module):
    def __init__(self, dim: int, n_groups: int = 4):
        super().__init__()
        self.layers = nn.Sequential(                
            nn.GroupNorm(num_groups=n_groups, num_channels=dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=n_groups, num_channels=dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.layers(x)
    
class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, resnet_blocks: int = 4):
        super().__init__()
        self.resnets = nn.Sequential(*[ResNet(dim=in_channels) for _ in range(resnet_blocks)])
        self.down_conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.resnets(x)
        return self.down_conv(x)
    
class Attention(nn.Module):
    def __init__(self, latent_channels: int, n_heads: int):
        """
        input: (B, C, Z, Z)
        """
        super().__init__()
        self.n_heads = n_heads
        self.w_qkv = nn.Conv2d(latent_channels, 3 * latent_channels, kernel_size=1)
        self.wo = nn.Conv2d(latent_channels, latent_channels, kernel_size=1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # c = nh hs, attn over every latent pixel
        assert (x.shape[1] % self.n_heads == 0), f"Latent Channels {x.shape[1]} must be divisible by n_heads {self.n_heads}"
        q, k, v = rearrange(self.w_qkv(x), "b (k nh hs) h w -> k b nh (h w) hs", k=3, nh=self.n_heads).unbind(0)
        attn = rearrange(F.scaled_dot_product_attention(q, k, v), "b nh (h w) hs -> b (nh hs) h w", w=x.shape[-1])
        return x + self.wo(attn)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, channels: tuple[int, ...], z_channels: int, n_heads: int = 1, resnet_blocks: int = 4, bottleneck_layers: int = 4):
        """
        input: (B, C, H, W)
        output: (B, z_channels, H/f, W/f)
        """
        super().__init__()
        self.downsample = nn.Sequential(
            nn.Conv2d(3, channels[0], kernel_size=3, padding=1),
            *[DownBlock(channels[i], channels[i+1], resnet_blocks) for i in range(len(channels) - 1)]
        )
    
        self.bottleneck = nn.Sequential(
            *[
                nn.Sequential(Attention(channels[-1], n_heads), ResNet(channels[-1]))
                for _ in range(bottleneck_layers)
            ], 
            nn.Conv2d(channels[-1], z_channels, kernel_size=3, padding=1)
        )
        
        self.mu_proj = nn.Conv2d(z_channels, z_channels, kernel_size=3, padding=1)
        self.logvar_proj = nn.Conv2d(z_channels, z_channels, kernel_size=3, padding=1)
        
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor]:
        x = self.downsample(x)
        x = self.bottleneck(x)
        return self.mu_proj(x), self.logvar_proj(x)

In [247]:
class UpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, resnet_blocks: int = 4):
        super().__init__()
        self.resnets = nn.Sequential(*[ResNet(dim=in_channels) for _ in range(resnet_blocks)])
        self.up_conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.resnets(x)
        return self.up_conv(x)
    
class Decoder(nn.Module):
    def __init__(self, channels: tuple[int, ...], z_channels: int,  resnet_blocks: int = 4):
        super().__init__()
        self.upsample = nn.Sequential(
                    nn.ConvTranspose2d(z_channels, latent_channels[-1], kernel_size=3, stride=2, padding=1, output_padding=1),
                    *[UpBlock(channels[i], channels[i-1], resnet_blocks) for i in range((len(channels) - 1), 0, -1)],
                    nn.Conv2d(channels[0], 3, kernel_size=3, padding=1) # RGB is 3 output channels
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.upsample(x)

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_channels: tuple[int, ...], z_channels: int, n_heads: int, resnet_blocks: int = 4, bottleneck_layers: int = 4):
        """
        input: (B, C, H, W)
        latent: (B, z_channels, H/f, W/f), f = 2^len(latent_channels)
        """
        super().__init__()
        self.encoder = Encoder(latent_channels, z_channels, n_heads, resnet_blocks, bottleneck_layers)
        self.decoder = Decoder(latent_channels, z_channels)
    
    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, ...]:
        mu, logvar = self.encoder(x)
        reparam = mu + torch.randn_like(logvar) * torch.exp(0.5 * logvar)
        return reparam, mu, logvar
    
    def decode(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(x)

    def forward(self, x: torch.Tensor, beta: float = 1.0, fft_weight: float = 1.0) -> tuple[torch.Tensor, ...]:
        latent, mu, logvar = self.encode(x)
        output = self.decode(latent)
        
        if self.training is False:
            loss = None
        else:
            recon_loss = F.l1_loss(input=output, target=x)
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / self.config.batch_size
            lpips_loss = lpips_loss_function(x, output).mean()
            # fft_loss = F.l1_loss(input=torch.fft.rfft2(output, dim=(-2, -1)).abs(), target=torch.fft.rfft2(x, dim=(-2, -1)).abs())
            
            loss = recon_loss + lpips_loss + (beta) * kl_loss # + (fft_weight) * fft_loss
        return output, loss

In [208]:
data = torch.load("minidata/smallImageTensors.pt", weights_only=True).to(device) # pass through VAE
data.shape

torch.Size([16, 3, 256, 256])

In [164]:
latent_channels = [4, 8, 16, 32]
z_channels = 8

In [169]:
enc = Encoder(latent_channels, z_channels, n_heads=2).to(device)

In [172]:
enc.bottleneck(enc.downsample(data)).shape

torch.Size([16, 8, 32, 32])

In [250]:
model = VAE(
    latent_channels=[4, 8, 16, 32],
    z_channels=8,
    n_heads=2
).to(device)

In [ ]:
model(data, beta=0.1)